# Training the Trader Financial Resilience models

This notebook trains the two models used by the FastAPI backend:

1. **`replacement_cost_model`** — predicts a product's next wholesale unit cost from its last 5 purchase costs.
2. **`risk_classifier_model`** — predicts the probability that a trader is eating into their working capital.

**On the data:** no public dataset exists at the granularity this project needs (trader-level purchase histories, or trader-level cash-flow risk labels). Real sources like the World Bank Enterprise Surveys, FSD Kenya's FinAccess MSE Tracker, and CSA/KNBS CPI releases give real macro trends and firm-level snapshots, but not the granular time series a model like this needs to train on.

So this notebook builds a **calibrated synthetic dataset**: purchase-cost drift is set in the same range as real published Ethiopia/Kenya food & staples inflation (roughly 4-15% annualized), with realistic noise and occasional price shocks. 

**Swap in real data later:** if your team collects real Purchase or Transaction rows (from the backend, or a partner trader), replace the two `generate_*` functions below with a loader for that CSV, keeping the same column names - everything after that point works unchanged.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, f1_score, classification_report
import joblib

RNG = np.random.default_rng(42)
N_LAG = 5  # number of past purchase costs used as features

## 1. Replacement-cost model — generate synthetic purchase data

In [ ]:
def generate_purchase_data(n_products=300, n_cycles=24):
    """Simulates monthly restocking cycles per product: a cost that drifts
    upward (inflation), has everyday noise, and occasionally jumps (a supply
    shock) — the exact pattern that erodes a trader's capital if they don't
    reprice."""
    rows = []
    for product_id in range(n_products):
        base_cost = RNG.uniform(20, 500)
        monthly_drift = RNG.uniform(0.003, 0.012)  # ~4-15% annualized
        cost = base_cost
        history = []
        for cycle in range(n_cycles):
            shock = RNG.normal(0, 0.02)
            if RNG.random() < 0.05:
                shock += RNG.uniform(0.05, 0.20)
            cost = max(1.0, cost * (1 + monthly_drift + shock))
            history.append(cost)
            if len(history) > N_LAG:
                features = history[-(N_LAG + 1):-1]
                target = history[-1]
                rows.append({
                    "product_id": product_id,
                    **{f"lag_{i+1}": v for i, v in enumerate(features)},
                    "next_cost": target,
                })
    return pd.DataFrame(rows)


purchase_df = generate_purchase_data()
print("Purchase training rows:", len(purchase_df))
purchase_df.head()

### Train and evaluate the replacement-cost regressor

In [ ]:
feature_cols = [f"lag_{i+1}" for i in range(N_LAG)]
X = purchase_df[feature_cols]
y = purchase_df["next_cost"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

replacement_cost_model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
replacement_cost_model.fit(X_train, y_train)

preds = replacement_cost_model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"MAE: {mae:.2f}  (average cost in data: {y.mean():.2f})")

In [ ]:
joblib.dump(replacement_cost_model, "replacement_cost_model.pkl")
print("Saved replacement_cost_model.pkl")

## 2. Risk classifier — generate synthetic trader financial data

In [ ]:
def generate_trader_financial_data(n_traders=2000):
    """Features match exactly what the backend sends at inference time
    (see app/services/prediction.py): income, expenses, withdrawals,
    open_credit, cash_on_hand. Label: 1 = at risk of eating into working
    capital, based on a transparent rule used only to label the synthetic
    training data (the model never sees this rule at inference time)."""
    rows = []
    for _ in range(n_traders):
        income = RNG.uniform(500, 20000)
        expense_ratio = RNG.uniform(0.4, 1.3)
        withdrawal_ratio = RNG.uniform(0.0, 0.4)
        expenses = income * expense_ratio * RNG.uniform(0.8, 1.2)
        withdrawals = income * withdrawal_ratio * RNG.uniform(0.8, 1.2)
        open_credit = RNG.uniform(0, income * 0.6)
        cash_on_hand = max(0, income - expenses - withdrawals + RNG.normal(0, income * 0.1))

        risk_signal = (expenses + withdrawals + 0.5 * open_credit) / max(income, 1)
        label = 1 if risk_signal > 1.05 else 0

        rows.append({
            "income": income, "expenses": expenses, "withdrawals": withdrawals,
            "open_credit": open_credit, "cash_on_hand": cash_on_hand, "at_risk": label,
        })
    return pd.DataFrame(rows)


risk_df = generate_trader_financial_data()
print("Risk training rows:", len(risk_df))
risk_df["at_risk"].value_counts(normalize=True)

### Train and evaluate the risk classifier

In [ ]:
risk_features = ["income", "expenses", "withdrawals", "open_credit", "cash_on_hand"]
Xr, yr = risk_df[risk_features], risk_df["at_risk"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42, stratify=yr)

risk_classifier_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
risk_classifier_model.fit(Xr_train, yr_train)

yr_pred = risk_classifier_model.predict(Xr_test)
print(f"F1 score: {f1_score(yr_test, yr_pred):.3f}\n")
print(classification_report(yr_test, yr_pred))

In [ ]:
joblib.dump(risk_classifier_model, "risk_classifier_model.pkl")
print("Saved risk_classifier_model.pkl")

## 3. Integrate into the backend

Copy both `.pkl` files from this notebook's folder into the backend's `app/ml/` folder:

```bash
cp replacement_cost_model.pkl risk_classifier_model.pkl /path/to/trader_pos/app/ml/
```

Restart the API (`uvicorn main:app --reload`). `app/ml/ml_inference_service.py` loads both files automatically on startup - no other code changes needed. Test with:

- `GET /predictions/price-recommendation/{product_id}`
- `GET /predictions/risk-score/{trader_id}`
